In [ ]:
# -*- coding: utf-8 -*-
"""
5-fold cross-validation of CLIP-Base on the WAD20 dataset
=========================================================

WAD20 (released with Eisbach et al. 2023): 540 above-the-fold website
screenshots with individual ratings, averaged per page here.
https://doi.org/10.1145/3569889

Within each fold the model is trained with the phased protocol (head,
then last four encoder blocks, then the full network; 10 epochs per phase
with early stopping). Ratings are z-scored; Pearson r is invariant to
that rescaling. Fold metrics are aggregated and a bootstrap confidence
interval is computed over the pooled out-of-fold predictions.

Source of the WAD20 row in Table 5 of the paper.

Written for Google Colab. Expects the dataset archive at
/content/drive/MyDrive/datasets/WAD20.zip.
"""

import os
import random
import json
from dataclasses import dataclass
from datetime import datetime
import gc

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torch import amp

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import KFold

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Setup
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = torch.cuda.is_available()
print(f"Device: {device}")

# ============================================================
# Config
# ============================================================
@dataclass
class Config:
    # Paths
    WAD20_PATH: str = "/content/WAD20"
    CACHE_DIR: str = "/content/cache_wad20"
    RESULTS_DIR: str = "/content/drive/MyDrive/results/vit_benchmark/wad20_5fold"

    # Rating column
    RATING_COL: str = "aesthetics"  # Options: aesthetics, visualAppeal, visawi_overall

    # Model
    HF_MODEL: str = "openai/clip-vit-base-patch16"
    IMG_SIZE: int = 224
    BASE_LR: float = 3e-6
    HEAD_LR: float = 2e-4
    WEIGHT_DECAY: float = 0.01
    BATCH_SIZE: int = 16
    EPOCHS_HEAD: int = 10
    EPOCHS_PARTIAL: int = 10
    EPOCHS_FULL: int = 10
    WARMUP_RATIO: float = 0.1

    # CV
    N_FOLDS: int = 5
    SEED: int = 42

    # Eval
    USE_TTA: bool = False

    def __post_init__(self):
        from google.colab import drive
        drive.mount('/content/drive')

        if not os.path.exists(self.WAD20_PATH):
            os.makedirs(self.WAD20_PATH, exist_ok=True)
            os.system(f'unzip -q /content/drive/MyDrive/datasets/WAD20.zip -d {self.WAD20_PATH}')

        for d in [self.CACHE_DIR, self.RESULTS_DIR]:
            os.makedirs(d, exist_ok=True)

CFG = Config()

# ============================================================
# Logging
# ============================================================
class Logger:
    def __init__(self):
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.log_file = os.path.join(CFG.RESULTS_DIR, f"wad20_5fold_{ts}.log")

    def log(self, msg):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] {msg}")
        with open(self.log_file, "a") as f:
            f.write(f"[{ts}] {msg}\n")

logger = Logger()

# ============================================================
# Data Loading
# ============================================================
def load_wad20_data():
    logger.log("Loading WAD20 dataset...")

    # Find survey file
    survey_files = [
        os.path.join(CFG.WAD20_PATH, "surveyData.csv"),
        os.path.join(CFG.WAD20_PATH, "WAD20", "surveyData.csv"),
    ]
    survey_file = next((f for f in survey_files if os.path.exists(f)), None)
    if not survey_file:
        raise FileNotFoundError("WAD20 surveyData.csv not found")

    # Find image directory
    image_dirs = [
        os.path.join(CFG.WAD20_PATH, "data"),
        os.path.join(CFG.WAD20_PATH, "WAD20", "data"),
    ]
    image_dir = next((d for d in image_dirs if os.path.exists(d)), None)
    if not image_dir:
        raise FileNotFoundError("WAD20 image directory not found")

    logger.log(f"  Survey: {survey_file}")
    logger.log(f"  Images: {image_dir}")

    # Load raw data
    df_raw = pd.read_csv(survey_file)
    logger.log(f"  Raw ratings: {len(df_raw)}")

    # Aggregate per image
    df = df_raw.groupby("website").agg({
        CFG.RATING_COL: ["mean", "std", "count"]
    }).reset_index()
    df.columns = ["website", "rating", "rating_std", "n_ratings"]

    # Build paths
    df["screenshot_path"] = df["website"].apply(lambda w: os.path.join(image_dir, w))
    df["stimulusId"] = df["website"]

    # Filter existing
    missing = (~df["screenshot_path"].apply(os.path.exists)).sum()
    if missing > 0:
        logger.log(f"  Warning: {missing} images not found")
    df = df[df["screenshot_path"].apply(os.path.exists)].reset_index(drop=True)

    logger.log(f"  Final: {len(df)} samples")
    logger.log(f"  Rating range: [{df['rating'].min():.2f}, {df['rating'].max():.2f}]")
    logger.log(f"  Avg ratings/image: {df['n_ratings'].mean():.1f}")
    logger.log(f"  Using column: {CFG.RATING_COL}")

    return df

# ============================================================
# Dataset
# ============================================================
def letterbox_square(img, size):
    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (128, 128, 128))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((size, size), Image.BICUBIC)


def get_cache_path(src_path):
    basename = os.path.splitext(os.path.basename(src_path))[0]
    return os.path.join(CFG.CACHE_DIR, f"wad20_{basename}_{CFG.IMG_SIZE}.jpg")


class WAD20Dataset(Dataset):
    def __init__(self, df, mean, std):
        self.df = df.reset_index(drop=True)
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_path = get_cache_path(row["screenshot_path"])

        if os.path.exists(cache_path):
            img = Image.open(cache_path).convert("RGB")
        else:
            img = Image.open(row["screenshot_path"]).convert("RGB")
            img = letterbox_square(img, CFG.IMG_SIZE)
            img.save(cache_path, "JPEG", quality=95)
            img = Image.open(cache_path).convert("RGB")

        x = TF.to_tensor(img)
        x = (x - torch.tensor(self.mean).view(-1,1,1)) / torch.tensor(self.std).view(-1,1,1)
        y = torch.tensor([row["rating_norm"]], dtype=torch.float32)

        return {"pixel_values": x, "labels": y, "idx": idx}

# ============================================================
# Metrics
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "r": float(pearsonr(y_pred, y_true)[0]),
        "rho": float(spearmanr(y_pred, y_true)[0]),
        "rmse": float(np.sqrt(np.mean((y_pred - y_true)**2))),
    }


def bootstrap_ci(y_true, y_pred, metric_fn, n=2000, alpha=0.05):
    rng = np.random.default_rng(42)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.percentile(vals, [100*alpha/2, 100*(1-alpha/2)])
# ============================================================
# Model
# ============================================================
def load_fresh_model():
    model = AutoModelForImageClassification.from_pretrained(
        CFG.HF_MODEL, num_labels=1, problem_type="regression", ignore_mismatched_sizes=True
    )
    model.to(device)

    try:
        proc = AutoImageProcessor.from_pretrained(CFG.HF_MODEL)
        mean, std = proc.image_mean, proc.image_std
    except:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    return model, mean, std


@torch.no_grad()
def predict(model, loader, use_tta=True):
    model.eval()
    preds, indices = [], []
    with amp.autocast('cuda', enabled=use_cuda):
        for batch in loader:
            x = batch["pixel_values"].to(device)
            if use_tta:
                out = (model(x).logits + model(torch.flip(x, [-1])).logits) / 2
            else:
                out = model(x).logits
            preds.append(out.squeeze(1).cpu().numpy())
            indices.extend(batch["idx"].numpy().tolist())
    return np.concatenate(preds), indices

# ============================================================
# Training
# ============================================================
def loss_function(pred, target):
    return F.mse_loss(pred, target)


def train_fold(model, train_loader, val_loader, fold_num):
    """Train one fold with 3-phase training."""

    def freeze_all(m, freeze=True):
        for p in m.parameters():
            p.requires_grad = not freeze

    def get_head_params(m):
        return [n for n, _ in m.named_parameters()
                if any(n.startswith(p) for p in ["classifier", "head", "fc", "score"])]

    def unfreeze_head(m):
        freeze_all(m, True)
        for n, p in m.named_parameters():
            if n in get_head_params(m):
                p.requires_grad = True

    def unfreeze_last_k(m, k=4):
        freeze_all(m, True)
        for n, p in m.named_parameters():
            if n in get_head_params(m):
                p.requires_grad = True
        if hasattr(m, "vision_model") and hasattr(m.vision_model, "encoder"):
            blocks = list(m.vision_model.encoder.layers)
            for i in range(max(0, len(blocks)-k), len(blocks)):
                for p in blocks[i].parameters():
                    p.requires_grad = True

    def unfreeze_all(m):
        freeze_all(m, False)

    best_r = -float("inf")
    best_state = None

    def run_phase(name, epochs, lr, unfreeze_fn):
        nonlocal best_r, best_state

        unfreeze_fn(model)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=lr, weight_decay=CFG.WEIGHT_DECAY)
        steps = epochs * len(train_loader)
        sch = get_cosine_schedule_with_warmup(opt, int(CFG.WARMUP_RATIO * steps), steps)
        scaler = amp.GradScaler(enabled=use_cuda)
        patience = 0

        for epoch in range(1, epochs + 1):
            model.train()
            for batch in train_loader:
                x = batch["pixel_values"].to(device)
                y = batch["labels"].to(device)

                with amp.autocast('cuda', enabled=use_cuda):
                    loss = loss_function(model(x).logits, y)

                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
                sch.step()

            # Validate
            y_pred, _ = predict(model, val_loader, use_tta=False)
            y_true = val_loader.dataset.df["rating_norm"].values
            r = pearsonr(y_pred, y_true)[0]

            if r > best_r:
                best_r = r
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 5:
                    break

        if best_state:
            model.load_state_dict(best_state)

    # 3-phase training
    run_phase("P1", CFG.EPOCHS_HEAD, CFG.HEAD_LR, unfreeze_head)
    run_phase("P2", CFG.EPOCHS_PARTIAL, CFG.BASE_LR, lambda m: unfreeze_last_k(m, 4))
    run_phase("P3", CFG.EPOCHS_FULL, CFG.BASE_LR, unfreeze_all)

    return model

# ============================================================
# Main: 5-Fold CV
# ============================================================
def main():
    logger.log("="*60)
    logger.log("5-FOLD CROSS-VALIDATION ON WAD20 DATASET")
    logger.log("540 website screenshots (Eisbach et al., 2023)")
    logger.log("="*60)

    # Load data
    df = load_wad20_data()

    # Precache all images
    logger.log("Precaching images...")
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
        cache_path = get_cache_path(row["screenshot_path"])
        if not os.path.exists(cache_path):
            img = Image.open(row["screenshot_path"]).convert("RGB")
            img = letterbox_square(img, CFG.IMG_SIZE)
            img.save(cache_path, "JPEG", quality=95)

    # Get normalization stats
    _, mean, std = load_fresh_model()

    # Z-score the ratings; Pearson r is invariant to this affine rescaling
    mu, sd = df["rating"].mean(), df["rating"].std()
    df["rating_norm"] = (df["rating"] - mu) / sd
    logger.log(f"Rating normalization: mean={mu:.3f}, std={sd:.3f}")

    # 5-Fold CV
    kfold = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

    fold_results = []
    all_predictions = np.zeros(len(df))
    all_true = df["rating_norm"].values

    for fold, (train_idx, test_idx) in enumerate(kfold.split(df), 1):
        logger.log(f"\n{'='*60}")
        logger.log(f"FOLD {fold}/{CFG.N_FOLDS}")
        logger.log(f"{'='*60}")
        logger.log(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

        # Split train into train/val
        n_val = int(len(train_idx) * 0.15)
        np.random.shuffle(train_idx)
        val_idx = train_idx[:n_val]
        train_idx_final = train_idx[n_val:]

        df_train = df.iloc[train_idx_final].reset_index(drop=True)
        df_val = df.iloc[val_idx].reset_index(drop=True)
        df_test = df.iloc[test_idx].reset_index(drop=True)

        logger.log(f"  Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

        # Load fresh model
        model, mean, std = load_fresh_model()

        # Create datasets
        train_ds = WAD20Dataset(df_train, mean, std)
        val_ds = WAD20Dataset(df_val, mean, std)
        test_ds = WAD20Dataset(df_test, mean, std)

        train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                                  num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                               num_workers=0, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                                num_workers=0, pin_memory=True)

        # Train
        model = train_fold(model, train_loader, val_loader, fold)

        # Test
        y_pred, pred_indices = predict(model, test_loader, use_tta=CFG.USE_TTA)
        y_true = df_test["rating_norm"].values

        # Store predictions
        for i, pred_idx in enumerate(pred_indices):
            all_predictions[test_idx[pred_idx]] = y_pred[i]

        # Compute metrics
        metrics = compute_metrics(y_true, y_pred)

        fold_results.append({
            "fold": fold,
            "r": metrics["r"],
            "rho": metrics["rho"],
            "rmse": metrics["rmse"],
            "n_test": len(test_idx)
        })

        logger.log(f"\nFold {fold} Overall Results:")
        logger.log(f"  r      = {metrics['r']:.4f}")
        logger.log(f"  ρ      = {metrics['rho']:.4f}")
        logger.log(f"  RMSE   = {metrics['rmse']:.4f}")

        # Cleanup
        del model, train_ds, val_ds, test_ds
        del train_loader, val_loader, test_loader
        torch.cuda.empty_cache()
        gc.collect()

    # Aggregate results
    logger.log("\n" + "="*60)
    logger.log("5-FOLD CV RESULTS - WAD20")
    logger.log("="*60)

    r_scores = [f["r"] for f in fold_results]
    rho_scores = [f["rho"] for f in fold_results]

    logger.log(f"\nPer-fold r: {[f'{x:.3f}' for x in r_scores]}")
    logger.log(f"Per-fold ρ: {[f'{x:.3f}' for x in rho_scores]}")

    mean_r = np.mean(r_scores)
    std_r = np.std(r_scores)
    mean_rho = np.mean(rho_scores)
    std_rho = np.std(rho_scores)

    logger.log(f"\nAggregated (mean ± std):")
    logger.log(f"  r   = {mean_r:.4f} ± {std_r:.4f}")
    logger.log(f"  ρ   = {mean_rho:.4f} ± {std_rho:.4f}")

    # Overall metrics
    overall_r = pearsonr(all_predictions, all_true)[0]
    overall_rho = spearmanr(all_predictions, all_true)[0]

    logger.log(f"\nOverall (all predictions combined):")
    logger.log(f"  r   = {overall_r:.4f}")
    logger.log(f"  ρ   = {overall_rho:.4f}")

    # Bootstrap CI on overall predictions
    r_ci = bootstrap_ci(all_true, all_predictions, lambda t, p: pearsonr(p, t)[0])
    logger.log(f"\nBootstrap 95% CI (overall):")
    logger.log(f"  r:   [{r_ci[0]:.3f}, {r_ci[1]:.3f}]")

    # Save results
    results = {
        "dataset": "WAD20",
        "rating_column": CFG.RATING_COL,
        "fold_results": fold_results,
        "mean_r": mean_r,
        "std_r": std_r,
        "mean_rho": mean_rho,
        "std_rho": std_rho,
        "overall_r": overall_r,
        "overall_rho": overall_rho,
        "n_samples": len(df),
        "n_folds": CFG.N_FOLDS,
        "config": {
            "epochs": f"{CFG.EPOCHS_HEAD}/{CFG.EPOCHS_PARTIAL}/{CFG.EPOCHS_FULL}",
            "base_lr": CFG.BASE_LR,
            "head_lr": CFG.HEAD_LR,
        }
    }

    results_path = os.path.join(CFG.RESULTS_DIR, "wad20_5fold_results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    logger.log(f"\nSaved: {results_path}")

    # Save predictions
    pred_df = df[["stimulusId", "rating"]].copy()
    pred_df["prediction_norm"] = all_predictions
    pred_df["prediction"] = all_predictions * sd + mu
    pred_path = os.path.join(CFG.RESULTS_DIR, "wad20_5fold_predictions.csv")
    pred_df.to_csv(pred_path, index=False)
    logger.log(f"Saved predictions: {pred_path}")

    logger.log("\n" + "="*60)
    logger.log("DONE")
    logger.log("="*60)

    return results


if __name__ == "__main__":
    main()

Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[11:16:29] ============================================================
[11:16:29] 5-FOLD CROSS-VALIDATION ON WAD20 DATASET
[11:16:29] Delitzas et al. (2023) - 540 website screenshots
[11:16:29] ============================================================
[11:16:29] Loading WAD20 dataset...
[11:16:29]   Survey: /content/WAD20/surveyData.csv
[11:16:29]   Images: /content/WAD20/data
[11:16:29]   Raw ratings: 6195
[11:16:29]   Final: 540 samples
[11:16:29]   Rating range: [1.75, 8.00]
[11:16:29]   Avg ratings/image: 11.5
[11:16:29]   Using column: aesthetics
[11:16:29] Precaching images...


  0%|          | 0/540 [00:00<?, ?it/s]

Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[11:16:38] Rating normalization: mean=4.871, std=1.255
[11:16:38] 
[11:16:38] FOLD 1/5
[11:16:38] ============================================================
[11:16:38] Train: 432, Test: 108
[11:16:38]   Train: 368, Val: 64, Test: 108


Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

[11:20:28] 
Fold 3 Overall Results:
[11:20:28]   r      = 0.6932
[11:20:28]   ρ      = 0.7126
[11:20:28]   RMSE   = 0.7089
[11:20:29] 
[11:20:29] FOLD 4/5
[11:20:29] ============================================================
[11:20:29] Train: 432, Test: 108
[11:20:29]   Train: 368, Val: 64, Test: 108


Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[11:21:51] 
Fold 4 Overall Results:
[11:21:51]   r      = 0.7444
[11:21:51]   ρ      = 0.7109
[11:21:51]   RMSE   = 0.6620
[11:21:51] 
[11:21:51] FOLD 5/5
[11:21:51] ============================================================
[11:21:51] Train: 432, Test: 108
[11:21:51]   Train: 368, Val: 64, Test: 108


Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[11:23:10] 
Fold 5 Overall Results:
[11:23:10]   r      = 0.5133
[11:23:10]   ρ      = 0.5437
[11:23:10]   RMSE   = 0.8649
[11:23:10] 
[11:23:10] 5-FOLD CV RESULTS - WAD20
[11:23:10] ============================================================
[11:23:10] 
Per-fold r: ['0.618', '0.715', '0.693', '0.744', '0.513']
[11:23:10] Per-fold ρ: ['0.584', '0.730', '0.713', '0.711', '0.544']
[11:23:10] 
Aggregated (mean ± std):
[11:23:10]   r   = 0.6566 ± 0.0830
[11:23:10]   ρ   = 0.6561 ± 0.0769
[11:23:10] 
Overall (all predictions combined):
[11:23:10]   r   = 0.6367
[11:23:10]   ρ   = 0.6401
[11:23:11] 
Bootstrap 95% CI (overall):
[11:23:11]   r:   [0.588, 0.683]
[11:23:11] 
Saved: /content/drive/MyDrive/results/vit_benchmark/wad20_5fold/wad20_5fold_results.json
[11:23:11] Saved predictions: /content/drive/MyDrive/results/vit_benchmark/wad20_5fold/wad20_5fold_predictions.csv
[11:23:11] 
[11:23:11] DONE
[11:23:11] ============================================================
